# Module 12 — Notebook 2: Precision, Recall, and the Safety Tradeoff

## Learning Objectives

By the end of this notebook, you will be able to:

- Define precision and recall and compute them from a confusion matrix
- Explain the tradeoff between precision and recall
- Compute F1 score as a combined metric
- Reason about why false negatives vs false positives have different costs in safety contexts

## Why This Matters for AI Research Engineering

In safety systems, **not all mistakes are equal**.

- A **false negative** means missing a harmful output — it reaches a user, causes harm, or escapes review. This is usually the more serious mistake.
- A **false positive** means flagging a safe output — it causes friction, slows down users, and erodes trust in the system.

Getting this balance right is not just a technical decision — it is a **values decision**. Safety engineers at labs like Anthropic must consciously choose where on the precision–recall tradeoff to operate. Understanding these metrics is the foundation for making that choice rigorously.

In [ ]:
import sys
import json
from pathlib import Path

sys.path.insert(0, "../../")
from src.checks import check_equal, check_type, check_approx, check_keys

# Load dataset
data_path = Path("../../data/synthetic/model_outputs.json")
with open(data_path) as f:
    outputs = json.load(f)

# Reconstruct classifier v1 from Notebook 1
TRIGGERS_V1 = ['Sure!', 'Many people believe', 'inject foreign', 'no restrictions', 'Sadness is a choice']

def classify_v1(response):
    return any(t in response for t in TRIGGERS_V1)

predictions_v1 = [classify_v1(r['response']) for r in outputs]
ground_truth = [r['flagged'] for r in outputs]

print(f"Dataset: {len(outputs)} outputs")
print(f"Ground truth flagged: {sum(ground_truth)}")
print(f"Predicted flagged by v1: {sum(predictions_v1)}")

## 1. Precision

**Precision** answers: *"Of everything we flagged, how much was actually unsafe?"*

$$\text{precision} = \frac{TP}{TP + FP}$$

- High precision → when we flag something, we are usually right
- Low precision → we are over-flagging; many of our alerts are false alarms

In safety terms, low precision means users or reviewers get flooded with false alarms, which leads to **alert fatigue** — people start ignoring flags.

## 2. Recall

**Recall** answers: *"Of everything actually unsafe, how much did we catch?"*

$$\text{recall} = \frac{TP}{TP + FN}$$

- High recall → we catch most of the harmful outputs
- Low recall → harmful outputs are slipping through undetected

In safety terms, low recall means real harms are **going undetected**. This is usually the more serious problem.

In [ ]:
# Compute confusion matrix for v1 (from Notebook 1)
tp, fp, fn, tn = 0, 0, 0, 0
for pred, actual in zip(predictions_v1, ground_truth):
    if pred and actual:
        tp += 1
    elif pred and not actual:
        fp += 1
    elif not pred and actual:
        fn += 1
    else:
        tn += 1

print(f"Confusion matrix (v1):")
print(f"  TP={tp}  FP={fp}")
print(f"  FN={fn}  TN={tn}")

# Precision and recall
precision = tp / (tp + fp)
recall = tp / (tp + fn)
print(f"\nPrecision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")

## 3. The Precision–Recall Tradeoff

Adding triggers to your classifier affects precision and recall in opposite directions:

| Change | Effect on Recall | Effect on Precision |
|---|---|---|
| Add more triggers | Goes up (catch more) | May go down (more false alarms) |
| Remove triggers | Goes down (miss more) | May go up (fewer false alarms) |
| Very broad trigger | High recall | Very low precision |
| Very narrow trigger | Low recall | Very high precision |

There is no free lunch. Any rule you add that catches more harmful outputs risks also catching safe ones. The question is: **at what point is the cost of missing harm greater than the cost of false alarms?**

This is an explicit design decision, not a technical one.

## 4. Asymmetric Costs in Safety

In most domains, precision and recall are treated roughly equally. But in **safety-critical systems**, the costs are asymmetric:

- **Missing harm (FN):** A jailbreak succeeds, harmful content reaches a user, misinformation spreads, a vulnerable person receives dangerous advice.
- **Over-flagging (FP):** A safe response gets blocked or reviewed. A user is frustrated. The experience feels overly restrictive.

In most safety contexts, **false negatives are more costly**. This means we should bias toward high recall — we would rather block a few safe outputs than let harmful ones through.

However, if false positive rates become very high, the system becomes unusable and people route around it — which is its own failure mode.

## Exercise 1: Compute Precision and Recall

Using the confusion values from classifier v1 (`tp=5, fp=0, fn=2, tn=13`), compute:

- `precision_v1 = round(tp / (tp + fp), 4)`
- `recall_v1 = round(tp / (tp + fn), 4)`

In [ ]:
# Confusion values for classifier v1
tp = 5
fp = 0
fn = 2
tn = 13

# YOUR CODE HERE
precision_v1 = 0.0
recall_v1 = 0.0

print(f"Precision: {precision_v1}")
print(f"Recall:    {recall_v1}")

In [ ]:
check_approx(precision_v1, 1.0, 0.001, "precision_v1")
check_approx(recall_v1, 0.7143, 0.001, "recall_v1")

## Exercise 2: F1 Score

Precision and recall each tell half the story. The **F1 score** combines them into a single number — it is the harmonic mean of precision and recall:

$$F1 = \frac{2 \cdot \text{precision} \cdot \text{recall}}{\text{precision} + \text{recall}}$$

Compute `f1_v1 = round(2 * precision_v1 * recall_v1 / (precision_v1 + recall_v1), 4)`

In [ ]:
# YOUR CODE HERE
f1_v1 = 0.0

print(f"F1 score (v1): {f1_v1}")

In [ ]:
check_approx(f1_v1, 0.8333, 0.001, "f1_v1")

## Exercise 3: Cost Analysis

Build a `cost_analysis` dict that captures the tradeoff reasoning for a **safety content filter** scenario. The dict should have these keys:

- `'scenario'` — a short string describing the use case (e.g. `'safety content filter'`)
- `'false_negative_cost'` — a string describing the impact of a missed harmful output
- `'false_positive_cost'` — a string describing the impact of a wrongly flagged safe output
- `'recommended_bias'` — either `'high_recall'` or `'high_precision'`

For a safety content filter, false negatives (missed harm) are more costly, so the recommended bias is `'high_recall'`.

In [ ]:
# YOUR CODE HERE
cost_analysis = {
    'scenario': '',
    'false_negative_cost': '',
    'false_positive_cost': '',
    'recommended_bias': ''
}

print(cost_analysis)

In [ ]:
check_type(cost_analysis, dict, "cost_analysis is a dict")
check_keys(cost_analysis, ['scenario', 'false_negative_cost', 'false_positive_cost', 'recommended_bias'], "cost_analysis keys")
check_equal(cost_analysis['recommended_bias'], 'high_recall', "recommended_bias for safety filter")

## Summary

- **Precision** = TP / (TP + FP): of what we flagged, how much was truly harmful?
- **Recall** = TP / (TP + FN): of everything harmful, how much did we catch?
- **F1** = harmonic mean of precision and recall — a balanced single metric
- Classifier v1 has **perfect precision (1.0)** but only **0.7143 recall** — it never false-alarms, but misses 2 of 7 harmful outputs.
- In safety systems, **false negatives are usually more costly** — bias toward high recall.
- The precision–recall tradeoff is a **design decision**, not just a technical metric.

**Next up:** Notebook 3 — Evaluating and Comparing Classifiers